In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import openpathsampling as paths
import aimmd 
import torch
import sys

In [ ]:
%run ../Toy_potentials.py
%run ../TIS_AIMMD_setup.py
%run ../../Resources/toy_plot_helpers.py

In [ ]:
interface_value = -8
int_val_int = int(100*interface_value)

In [ ]:
ResourceDirectoryPath = ""

In [ ]:
aimmd_store = aimmd.Storage(ResourceDirectoryPath+"Resources/aimmd_storage_pot_1_after_tis_1.h5", "r")

In [ ]:
# %%
model = aimmd_store.rcmodels["most_recent"]
# move model to current device:
if torch.cuda.is_available():
    model._device="cuda"
    model.nnet.to("cuda")
else :
    model._device = "cpu"
    model.nnet.to("cpu")

In [ ]:
pes = potential_1()

In [ ]:
x = np.linspace(pes.extent[0],pes.extent[1],300)
y = np.linspace(pes.extent[2],pes.extent[3],300)
x_2d, y_2d, U = pes.plot_pes(x,y)
# Step 2: Create the colormap
colormap = 'plasma'  # You can choose a different colormap from Matplotlib's colormaps

# Step 3: Generate the 2D image

plt.figure()
plt.contour(x_2d,y_2d,U, levels = pes.levels)
plt.colorbar(label='Z-axis value')  # Add a colorbar to show the intensity scale
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()

In [ ]:

integ = toys.LangevinBAOABIntegrator(dt=0.02, temperature=0.1, gamma=2.5)
options={
    'integ' : integ,
    'n_frames_max' : 5000,
    'n_steps_per_frame' : 1
}
toy_eng = toys.Engine(
    options=options,
    topology=pes.topology
)
toy_eng.initialized = True

template = pes.template(toy_eng)
toy_eng.current_snapshot = template
paths.PathMover.engine = toy_eng

# Collective variables to define the states
opA = paths.CoordinateFunctionCV(name="opA", f=pes.stable_interface_function, center=pes.state_A)
opB = paths.CoordinateFunctionCV(name="opB", f=pes.stable_interface_function, center=pes.state_B)

# State volumes in CV space
stateA = paths.CVDefinedVolume(opA, 0.0, pes.state_boundary).named('StateA')
stateB = paths.CVDefinedVolume(opB, 0.0, pes.state_boundary).named('StateB')
#

In [ ]:
# descriptor_transform = paths.FunctionCV('descriptor_transform', lambda s: s.coordinates[0], cv_wrap_numpy_array=True).with_diskcache()


In [ ]:
# Fake an initial trajectory
init_AB = paths.Trajectory(pes.simple_initial_path(100,toy_eng))


In [ ]:
TIS_Framework = AIMMD_TIS(toy_eng, model, stateA, stateB)

In [ ]:
storage = paths.Storage("simple_store_1.nc", "w", template=template)
storage.save(toy_eng)

In [ ]:
print(storage.engines)

In [ ]:
TIS_Framework.run_single_TIS(100, storage, init_AB, interface_value)

In [ ]:
trajectories = [step.active[0].trajectory for step in storage.steps[::10]]
plt.figure()
plt.contour(x_2d,y_2d,U, pes.levels)
plt.colorbar(label='Z-axis value') 
repcolordict = {0 : 'k-', 1 : 'r-', 2 : 'g-', 3 : 'b-', 4 : 'r-'}
for i, traj in enumerate(trajectories):
    plt.plot(traj.xyz[:,0,0], traj.xyz[:,0,1], repcolordict[i % 5])

states_plot = np.vectorize(CallableVolume(stateA))(x_2d, -y_2d)
states_plot += np.vectorize(CallableVolume(stateB))(x_2d, -y_2d)
plt.imshow(states_plot, extent=pes.extent, cmap="Blues",
            interpolation='nearest', vmin=0.0, vmax=2.0,
            aspect='auto')
 # Add a colorbar to show the intensity scale
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.show()